# Week 4 — CI/CD & Testing for ML

### Automate the path to production — and learn the tests that only ML needs

> **MLOps & Deployment: A From-Scratch Engineering Codex**
> Part of the applied-systems track. Like the rest of this curriculum, every
> abstraction here is *built before it is used*: we re-implement the core of each
> MLOps tool in plain Python/NumPy first, then map what we built onto the
> industry-standard equivalent so the magic is gone by the time you reach it.

---

## Learning objectives

By the end of this notebook you will be able to:

1. Build a pipeline DAG executor with content-based caching from scratch.
2. Write the test types unique to ML: data validation, behavioral, and invariance tests.
3. Implement a champion/challenger gate that decides promotion automatically.
4. Assemble these into a CI pipeline that fails fast and only ships validated models.
5. Translate the whole thing into a GitHub Actions workflow.

## Prerequisites

- Weeks 0–3 (manifests, registry, packaging, serving).
- Basic familiarity with unit testing and Git.

## How to read this notebook

Each section has three layers:

- **🧠 Theory** — the systems problem and *why* it exists.
- **🔧 From scratch** — a minimal, dependency-light implementation you fully own.
- **🏭 In practice** — how the real tool (MLflow, Docker, K8s, etc.) does the same thing, and what it adds.

Run cells top-to-bottom. Everything is self-contained and works on a CPU-only laptop.

In [1]:
# --- Standard setup ---------------------------------------------------------
from __future__ import annotations
import os, sys, json, time, math, hashlib, random, pathlib, tempfile, shutil
from dataclasses import dataclass, field, asdict
from typing import Any, Callable

import numpy as np

np.random.seed(0)
random.seed(0)

# A scratch workspace that we clean up between runs of this notebook.
WORK = pathlib.Path(tempfile.mkdtemp(prefix="mlops_week_"))
print("Working directory for this notebook:", WORK)

# --- Content-addressable hashing (introduced & explained in Week 0) ---------
# Re-defined here so each notebook is fully self-contained and runnable alone.
def hash_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()[:12]

def hash_obj(obj) -> str:
    canonical = json.dumps(obj, sort_keys=True, separators=(",", ":"), default=str)
    return hash_bytes(canonical.encode())

def hash_array(a: np.ndarray) -> str:
    return hash_bytes(a.tobytes() + str(a.shape).encode() + str(a.dtype).encode())

def capture_environment() -> dict:
    import platform
    return {"python": sys.version.split()[0], "platform": platform.platform(),
            "numpy": np.__version__}

Working directory for this notebook: /tmp/mlops_week_qs4c9twb


## 1. 🧠 Why ML needs more than software CI/CD

Classic CI/CD tests *code*: does it compile, do unit tests pass, does it deploy.
ML adds a second axis that software never had — **the model's behaviour depends on
data**, so code passing tests is necessary but wildly insufficient. You can have
perfect code and a model that is silently 20% worse than last week's.

So ML CI/CD validates **three things**, not one:

1. **Code** — the usual: lints, unit tests, builds. (Software CI.)
2. **Data** — schema, ranges, distribution, no leakage. (New.)
3. **Model** — accuracy *and* behaviour, vs. a baseline champion. (New.)

And it produces a different artifact: not just a binary, but a **validated,
registered model promoted to a stage** (Week 1's registry). This week wires the
whole loop together and automates the promotion decision you made by hand in Week 1.

## 2. 🔧 From scratch: a pipeline DAG with caching

A training pipeline is a directed acyclic graph of steps: `ingest → validate →
features → train → evaluate → register`. Two properties make it production-grade:

- **Determinism** — steps run in dependency order.
- **Caching** — a step is skipped if its inputs (and code) haven't changed
  (Week 0's content hashing — it keeps coming back).

We'll build a tiny DAG runner with exactly these properties.

In [2]:
@dataclass
class Step:
    name: str
    fn: Callable
    deps: list[str] = field(default_factory=list)

class Pipeline:
    def __init__(self):
        self.steps: dict[str, Step] = {}
        self.cache: dict[str, tuple[str, Any]] = {}   # step -> (input_hash, output)

    def step(self, name, deps=None):
        def deco(fn):
            self.steps[name] = Step(name, fn, deps or [])
            return fn
        return deco

    def _topo_order(self) -> list[str]:
        order, seen = [], set()
        def visit(n):
            if n in seen: return
            for d in self.steps[n].deps: visit(d)
            seen.add(n); order.append(n)
        for n in self.steps: visit(n)
        return order

    def run(self, context: dict | None = None, force: set | None = None):
        context = dict(context or {}); force = force or set()
        results, log = {}, []
        for name in self._topo_order():
            step = self.steps[name]
            dep_outputs = {d: results[d] for d in step.deps}
            # cache key = hash of (step source + dependency outputs + context)
            key = hash_obj({
                "src": step.fn.__doc__ or step.fn.__name__,
                "deps": {d: _safe_hash(o) for d, o in dep_outputs.items()},
                "ctx": {k: _safe_hash(v) for k, v in context.items()},
            })
            cached = self.cache.get(name)
            if cached and cached[0] == key and name not in force:
                results[name] = cached[1]; log.append((name, "CACHED"))
            else:
                out = step.fn(context, dep_outputs)
                self.cache[name] = (key, out)
                results[name] = out; log.append((name, "RAN"))
        return results, log

def _safe_hash(o):
    if isinstance(o, np.ndarray): return hash_array(o)
    try: return hash_obj(o)
    except Exception: return str(type(o))

print("Pipeline engine ready.")

Pipeline engine ready.


## 3. 🔧 Defining the ML pipeline steps

Each step is small and pure. Note the **data validation** and **evaluation** steps
— those are the ML-specific ones that ordinary CI lacks.

In [3]:
pipe = Pipeline()

@pipe.step("ingest")
def ingest(ctx, deps):
    "ingest_v1"
    rng = np.random.RandomState(ctx["seed"])
    X = rng.rand(ctx["n"], 2)
    y = X @ np.array([2.0, -1.0]) + 0.5 + ctx["noise"] * rng.randn(ctx["n"])
    return {"X": X, "y": y}

@pipe.step("validate_data", deps=["ingest"])
def validate_data(ctx, deps):
    "validate_v1"
    X, y = deps["ingest"]["X"], deps["ingest"]["y"]
    checks = {
        "no_nans": bool(not np.isnan(X).any() and not np.isnan(y).any()),
        "n_features_is_2": X.shape[1] == 2,
        "X_in_unit_range": bool(X.min() >= 0 and X.max() <= 1.0001),
        "enough_rows": len(X) >= 50,
    }
    if not all(checks.values()):
        raise ValueError(f"DATA VALIDATION FAILED: {checks}")
    return {"checks": checks, "n": len(X)}

@pipe.step("train", deps=["validate_data", "ingest"])
def train(ctx, deps):
    "train_v1"
    X, y = deps["ingest"]["X"], deps["ingest"]["y"]
    A = np.hstack([X, np.ones((len(X), 1))])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    return {"w": coef[:2], "b": float(coef[2])}

@pipe.step("evaluate", deps=["train", "ingest"])
def evaluate(ctx, deps):
    "evaluate_v1"
    X, y = deps["ingest"]["X"], deps["ingest"]["y"]
    w, b = deps["train"]["w"], deps["train"]["b"]
    mse = float(np.mean((X @ w + b - y) ** 2))
    return {"mse": mse}

ctx = {"seed": 0, "n": 300, "noise": 0.1}
results, log = pipe.run(ctx)
print("Run 1 execution log:")
for name, status in log: print(f"  {name:<14} {status}")
print("\nMSE:", round(results["evaluate"]["mse"], 5))
print("Data checks:", results["validate_data"]["checks"])

Run 1 execution log:
  ingest         RAN
  validate_data  RAN
  train          RAN
  evaluate       RAN

MSE: 0.01065
Data checks: {'no_nans': True, 'n_features_is_2': True, 'X_in_unit_range': True, 'enough_rows': True}


In [4]:
# Re-run with identical context: everything should be CACHED (nothing changed)
_, log2 = pipe.run(ctx)
print("Run 2 (no changes):")
for name, status in log2: print(f"  {name:<14} {status}")

# Change only the noise level: ingest changes -> everything downstream re-runs,
# but validate's *code* is unchanged so only data-dependent steps recompute.
ctx_b = {"seed": 0, "n": 300, "noise": 0.3}
_, log3 = pipe.run(ctx_b)
print("\nRun 3 (noise changed -> cache invalidated downstream):")
for name, status in log3: print(f"  {name:<14} {status}")

Run 2 (no changes):
  ingest         CACHED
  validate_data  CACHED
  train          CACHED
  evaluate       CACHED

Run 3 (noise changed -> cache invalidated downstream):
  ingest         RAN
  validate_data  RAN
  train          RAN
  evaluate       RAN


This caching is what makes iterating on a pipeline bearable: change the eval code
and you don't re-ingest gigabytes. It's the same content-hash idea from Week 0,
now governing *compute reuse* — exactly what DVC, Metaflow, and Kubeflow Pipelines
do under the hood.

## 4. 🧠 The tests that only ML needs

Beyond ordinary unit tests, ML systems need test categories that have no
equivalent in normal software. Three essential families:

- **Data validation tests** — schema, ranges, nulls, distribution. (We did this
  as a pipeline step; in CI it's a gate.)
- **Behavioral tests** — assert the model obeys *known properties* of the problem,
  independent of accuracy. E.g. "raising income should never lower the predicted
  credit limit." These are *invariance* and *directional* tests (from the CheckList
  methodology).
- **Champion/challenger tests** — the new model must beat (or at least not regress
  against) the current production model on a frozen evaluation set.

Let's implement behavioral and champion/challenger tests, since they're the
ML-native ones.

In [5]:
def make_predict(params):
    w, b = np.asarray(params["w"]), params["b"]
    return lambda X: np.atleast_2d(X) @ w + b

predict = make_predict(results["train"])

# --- Behavioral test: DIRECTIONAL expectation -------------------------------
# Our true relation is y = 2*x0 - 1*x1 + 0.5. So increasing x0 must increase y,
# and increasing x1 must DECREASE y. A model that violates this is wrong in a way
# accuracy alone might not reveal.
def test_directional(predict):
    base = np.array([[0.5, 0.5]])
    more_x0 = np.array([[0.9, 0.5]])
    more_x1 = np.array([[0.5, 0.9]])
    assert predict(more_x0)[0] > predict(base)[0], "x0 should increase prediction"
    assert predict(more_x1)[0] < predict(base)[0], "x1 should decrease prediction"
    return True

# --- Behavioral test: INVARIANCE --------------------------------------------
# Adding a tiny irrelevant perturbation shouldn't flip predictions wildly.
def test_invariance(predict, eps=1e-6):
    X = np.random.RandomState(3).rand(20, 2)
    delta = predict(X) - predict(X + eps)
    assert np.max(np.abs(delta)) < 1e-3, "model wildly sensitive to tiny noise"
    return True

print("directional test:", test_directional(predict))
print("invariance test  :", test_invariance(predict))

directional test: True
invariance test  : True


## 5. 🔧 The champion/challenger gate

This is the automated version of the promotion decision you made manually in
Week 1. The rule: a challenger may only be promoted if it beats the champion on a
**frozen holdout set** by at least a margin (to avoid promoting on noise).

In [6]:
# Frozen evaluation set — same for every model, forever (this is critical).
EVAL_RNG = np.random.RandomState(999)
X_eval = EVAL_RNG.rand(500, 2)
y_eval = X_eval @ np.array([2.0, -1.0]) + 0.5 + 0.1 * EVAL_RNG.randn(500)

def eval_mse(params):
    p = make_predict(params)
    return float(np.mean((p(X_eval) - y_eval) ** 2))

def champion_challenger_gate(champion, challenger, margin=0.0):
    """Return (promote: bool, report: dict). Challenger must beat champion by `margin`."""
    c_mse = eval_mse(champion)
    n_mse = eval_mse(challenger)
    improvement = (c_mse - n_mse) / c_mse
    promote = improvement > margin
    return promote, {
        "champion_mse": round(c_mse, 6),
        "challenger_mse": round(n_mse, 6),
        "relative_improvement": round(improvement, 4),
        "margin_required": margin,
        "decision": "PROMOTE" if promote else "REJECT",
    }

# Champion = a deliberately mediocre model; challenger = our trained one.
champion_params = {"w": np.array([1.5, -0.8]), "b": 0.4}   # worse fit
challenger_params = results["train"]                        # least-squares fit

promote, report = champion_challenger_gate(champion_params, challenger_params, margin=0.01)
print(json.dumps(report, indent=2))

# Now test the guard against noise: a challenger that's only trivially better.
tiny = {"w": champion_params["w"] * 1.001, "b": champion_params["b"]}
_, report2 = champion_challenger_gate(champion_params, tiny, margin=0.01)
print("\nTrivially-better challenger decision:", report2["decision"],
      "(improvement", report2["relative_improvement"], "< margin)")

{
  "champion_mse": 0.099742,
  "challenger_mse": 0.011646,
  "relative_improvement": 0.8832,
  "margin_required": 0.01,
  "decision": "PROMOTE"
}

Trivially-better challenger decision: REJECT (improvement 0.0033 < margin)


The margin matters enormously. Without it, you'd promote a new model every time it
was *0.0001%* better — which is just noise, and you'd churn production constantly.
The margin encodes *"is this improvement real and worth the deployment risk?"*

## 6. 🔧 Assembling the full CI pipeline

Now we chain it all into one `ci_run` that mirrors a real CI job: validate data →
train → behavioral tests → champion/challenger gate → register if and only if
everything passes. **Fail fast**: the first failed stage stops the pipeline and
nothing gets promoted.

In [7]:
def ci_run(ctx, champion_params, margin=0.01):
    stages = []
    def record(stage, ok, detail=""):
        stages.append((stage, "PASS" if ok else "FAIL", detail))
        return ok

    # Stage 1: pipeline (includes data validation, which raises on failure)
    try:
        results, _ = pipe.run(ctx, force={"ingest"})
        record("pipeline+data_validation", True, f"mse={results['evaluate']['mse']:.5f}")
    except Exception as e:
        record("pipeline+data_validation", False, str(e)); return False, stages

    challenger = results["train"]; predict = make_predict(challenger)

    # Stage 2: behavioral tests
    try:
        test_directional(predict); test_invariance(predict)
        record("behavioral_tests", True)
    except AssertionError as e:
        record("behavioral_tests", False, str(e)); return False, stages

    # Stage 3: champion/challenger gate
    promote, gate = champion_challenger_gate(champion_params, challenger, margin)
    record("champion_challenger", promote,
           f"{gate['challenger_mse']} vs {gate['champion_mse']} ({gate['decision']})")
    if not promote:
        return False, stages

    # Stage 4: register + promote (Week 1 registry would be called here)
    record("register_and_promote", True, "model promoted to Production")
    return True, stages

print("=== CI run on a healthy challenger ===")
ok, stages = ci_run({"seed": 0, "n": 300, "noise": 0.1}, champion_params)
for s, status, d in stages: print(f"  [{status}] {s:<28} {d}")
print("PIPELINE", "SUCCEEDED ✅" if ok else "FAILED ❌")

print("\n=== CI run where data validation fails (too few rows) ===")
ok2, stages2 = ci_run({"seed": 0, "n": 10, "noise": 0.1}, champion_params)
for s, status, d in stages2: print(f"  [{status}] {s:<28} {d}")
print("PIPELINE", "SUCCEEDED ✅" if ok2 else "FAILED ❌ (fail-fast: never reached training)")

=== CI run on a healthy challenger ===
  [PASS] pipeline+data_validation     mse=0.01065
  [PASS] behavioral_tests             
  [PASS] champion_challenger          0.011646 vs 0.099742 (PROMOTE)
  [PASS] register_and_promote         model promoted to Production
PIPELINE SUCCEEDED ✅

=== CI run where data validation fails (too few rows) ===
  [FAIL] pipeline+data_validation     DATA VALIDATION FAILED: {'no_nans': True, 'n_features_is_2': True, 'X_in_unit_range': True, 'enough_rows': False}
PIPELINE FAILED ❌ (fail-fast: never reached training)


The second run is the whole point: bad data **stops the pipeline before training**,
so a broken model can never reach the registry. In software CI a failing test
blocks a merge; in ML CI a failing *data* or *gate* check blocks a *promotion*.

## 7. 🏭 In practice: the same pipeline as GitHub Actions

Here's how the CI logic maps onto a real GitHub Actions workflow. The structure —
stages, fail-fast, artifact promotion — is identical to what you built.

In [8]:
GITHUB_ACTIONS = '''
name: ml-ci
on:
  pull_request:
    branches: [main]
  workflow_dispatch:

jobs:
  validate-and-train:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }

      - name: Install
        run: pip install -r requirements.txt

      # Stage 1: code tests (ordinary CI)
      - name: Unit tests
        run: pytest tests/unit -q

      # Stage 2: data validation (ML-specific) -- fails the job on bad data
      - name: Validate data
        run: python -m pipeline.validate_data

      # Stage 3: train + log to tracking server (Week 1)
      - name: Train
        run: python -m pipeline.train
        env:
          MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_URI }}

      # Stage 4: behavioral + champion/challenger gate -- the promotion decision
      - name: Behavioral tests & gate
        run: python -m pipeline.gate --margin 0.01

      # Stage 5: build & push the serving image (Week 2) only if gate passed
      - name: Build & push image
        if: success()
        run: |
          docker build -t $REGISTRY/linreg:${{ github.sha }} .
          docker push $REGISTRY/linreg:${{ github.sha }}

      # Stage 6: promote in the model registry (Week 1)
      - name: Promote to Staging
        if: success()
        run: python -m pipeline.promote --stage Staging --sha ${{ github.sha }}
'''
(WORK / "ml-ci.yml").write_text(GITHUB_ACTIONS)
print(GITHUB_ACTIONS)


name: ml-ci
on:
  pull_request:
    branches: [main]
  workflow_dispatch:

jobs:
  validate-and-train:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }

      - name: Install
        run: pip install -r requirements.txt

      # Stage 1: code tests (ordinary CI)
      - name: Unit tests
        run: pytest tests/unit -q

      # Stage 2: data validation (ML-specific) -- fails the job on bad data
      - name: Validate data
        run: python -m pipeline.validate_data

      # Stage 3: train + log to tracking server (Week 1)
      - name: Train
        run: python -m pipeline.train
        env:
          MLFLOW_TRACKING_URI: ${{ secrets.MLFLOW_URI }}

      # Stage 4: behavioral + champion/challenger gate -- the promotion decision
      - name: Behavioral tests & gate
        run: python -m pipeline.gate --margin 0.01

      # Stage 5: build & push the serving image (Week 2) only if g

Every stage corresponds to something you built:

| GitHub Actions stage | Built in |
|----------------------|----------|
| `Validate data` | this notebook (`validate_data` step) |
| `Train` + `MLFLOW_TRACKING_URI` | Week 1 tracking store |
| `Behavioral tests & gate` | this notebook (behavioral + champion/challenger) |
| `Build & push image` | Week 2 Dockerfile |
| `Promote to Staging` | Week 1 registry transitions |

The `if: success()` guards are the fail-fast property: a failed gate means no
image is built and nothing is promoted. **Continuous Deployment** (CD) is the same
file with an added job that promotes `Staging → Production` after, say, a
successful canary — which leads us straight into monitoring.

---

## 🎯 Exercises

Try these before moving on. They are deliberately open-ended — the goal is to
extend the machinery you just built, not to find a single right answer.

1. Add a *schema* test that fails if a new feature column appears or a column's dtype changes. Why is silent schema drift one of the most dangerous failures in ML CI?
2. Implement a *slice-based* champion/challenger gate: the challenger must not regress on ANY important data slice (e.g. each quartile of x0), even if its overall MSE improves. Why can an aggregate win hide a per-slice disaster?
3. Add a `training_time_budget` check that fails the pipeline if training exceeds N seconds. How does this protect a shared CI runner?
4. Make the gate *two-sided*: promote on improvement, but also alert (not fail) if the challenger is suspiciously *much* better than the champion. What real bug does 'too good' usually indicate? (Hint: leakage.)
5. Convert the `ci_run` stages into a JUnit-style XML report. Many CI systems render this natively — what do you gain by emitting a standard format?

---

## ✅ Recap — Week 4

- ML CI/CD validates three things — code, data, and model behaviour — and ships a promoted model, not just a binary.
- A pipeline is a cached DAG; content hashing decides what to recompute (Week 0 again).
- Behavioral tests (directional, invariance) check known properties independent of accuracy.
- The champion/challenger gate with a margin automates promotion and resists promoting on noise.
- Fail-fast means bad data or a failed gate blocks promotion before a broken model can ship; GitHub Actions just executes this structure.

### Coming up next

**Week 5 — Monitoring, Drift Detection & Observability.** The model is live. Now we watch it: we'll build statistical drift detectors (PSI, KS, a CUSUM change-point detector) from scratch, distinguish data drift from concept drift, and close the loop back to retraining.